In [ ]:
import pandas as pd
import numpy as np


# Load Data

df = pd.read_csv('/mnt/user-data/uploads/train.csv', index_col=0)

print("=" * 60)
print("ORIGINAL DATASET")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")



# PART A: Handle Missing Values

print("\n" + "=" * 60)
print("PART A: HANDLING MISSING VALUES")
print("=" * 60)

# Helper to strip units and convert to numeric
def strip_units(series, unit):
    return pd.to_numeric(series.str.replace(unit, '', regex=False).str.strip(), errors='coerce')

# Parse numeric values from string columns (needed for imputation stats)
mileage_num = strip_units(df['Mileage'], 'kmpl')
mileage_num = mileage_num.fillna(strip_units(df['Mileage'], 'km/kg'))
engine_num  = strip_units(df['Engine'], ' CC')
power_num   = strip_units(df['Power'], ' bhp')

# Mileage: 2 missing → impute with median (small count, median is robust)
mileage_median = mileage_num.median()
mileage_num = mileage_num.fillna(mileage_median)
print(f"Mileage  — 2 missing  → imputed with median ({mileage_median:.2f} kmpl)")

# Engine: 36 missing → impute with median
engine_median = engine_num.median()
engine_num = engine_num.fillna(engine_median)
print(f"Engine   — 36 missing → imputed with median ({engine_median:.0f} CC)")

# Power: 36 missing → impute with median
power_median = power_num.median()
power_num = power_num.fillna(power_median)
print(f"Power    — 36 missing → imputed with median ({power_median:.2f} bhp)")

# Seats: 38 missing → impute with mode (categorical-like integer)
seats_mode = df['Seats'].mode()[0]
df['Seats'] = df['Seats'].fillna(seats_mode)
print(f"Seats    — 38 missing → imputed with mode ({seats_mode:.0f})")

# New_Price: 5032/5847 (86%) missing → drop (too sparse to impute reliably)
df = df.drop(columns=['New_Price'])
print("New_Price — 5032/5847 missing (86%) → column DROPPED")

# Replace original string columns with clean numeric versions
df['Mileage'] = mileage_num.values
df['Engine']  = engine_num.values
df['Power']   = power_num.values

print(f"\nMissing values after Part A:\n{df.isnull().sum()}")


# ============================================================
# PART B: Units Removed (done above during Part A parsing)
# ============================================================
print("\n" + "=" * 60)
print("PART B: UNITS REMOVED — NUMERIC COLUMNS")
print("=" * 60)
print("  Mileage : removed 'kmpl' / 'km/kg'")
print("  Engine  : removed 'CC'")
print("  Power   : removed 'bhp'")
print(df[['Mileage', 'Engine', 'Power']].head())



# PART C: One-Hot Encoding — Fuel_Type and Transmissio
print("\n" + "=" * 60)
print("PART C: ONE-HOT ENCODING")
print("=" * 60)
print("Fuel_Type unique values :", df['Fuel_Type'].unique())
print("Transmission unique values:", df['Transmission'].unique())

df = pd.get_dummies(df, columns=['Fuel_Type', 'Transmission'], drop_first=False)

fuel_cols  = [c for c in df.columns if c.startswith('Fuel_Type_')]
trans_cols = [c for c in df.columns if c.startswith('Transmission_')]
print(f"\nNew Fuel_Type columns    : {fuel_cols}")
print(f"New Transmission columns : {trans_cols}")
print(f"\nSample:\n{df[fuel_cols + trans_cols].head()}")



# PART D: New Feature — Car_Age

print("\n" + "=" * 60)
print("PART D: NEW FEATURE — Car_Age = 2026 - Year")
print("=" * 60)
df['Car_Age'] = 2026 - df['Year']
print(df[['Year', 'Car_Age']].head(8).to_string())



# PART E: Pandas equivalents of dplyr operations

print("\n" + "=" * 60)
print("PART E: DATA MANIPULATION OPERATIONS")
print("=" * 60)

# --- SELECT ---
print("\n[SELECT] Choose key columns")
selected = df[['Name', 'Location', 'Year', 'Car_Age',
               'Kilometers_Driven', 'Mileage', 'Engine',
               'Power', 'Seats', 'Price']]
print(selected.head(5).to_string())

# --- FILTER ---
print("\n[FILTER] Diesel cars (Year > 2013) priced under 15 lakhs")
filtered = df[
    (df['Fuel_Type_Diesel'] == True) &
    (df['Year'] > 2013) &
    (df['Price'] < 15)
]
print(f"  Matching rows: {len(filtered)}")
print(filtered[['Name', 'Location', 'Year', 'Price']].head(5).to_string())

# --- RENAME ---
print("\n[RENAME] Kilometers_Driven → KM_Driven, Seats → Num_Seats")
renamed = selected.rename(columns={
    'Kilometers_Driven': 'KM_Driven',
    'Seats':             'Num_Seats'
})
print("  Columns:", renamed.columns.tolist())

# --- MUTATE ---
print("\n[MUTATE] Add Price_Per_Year = Price / Car_Age")
df['Price_Per_Year'] = (df['Price'] / df['Car_Age']).round(2)
print(df[['Name', 'Car_Age', 'Price', 'Price_Per_Year']].head(5).to_string())

# --- ARRANGE ---
print("\n[ARRANGE] Top 5 most expensive cars (Price descending)")
arranged = (df[['Name', 'Location', 'Year', 'Price']]
              .sort_values('Price', ascending=False))
print(arranged.head(5).to_string())

# --- SUMMARIZE with GROUP BY ---
print("\n[SUMMARIZE + GROUP BY] Average price & stats by Location")
summary = (df.groupby('Location')
             .agg(
                 Avg_Price = ('Price', 'mean'),
                 Max_Price = ('Price', 'max'),
                 Min_Price = ('Price', 'min'),
                 Count     = ('Price', 'count')
             )
             .round(2)
             .sort_values('Avg_Price', ascending=False)
             .reset_index())
print(summary.to_string())


# Save processed dataset

df.to_csv('/mnt/user-data/outputs/train_processed.csv', index=False)
print("\n" + "=" * 60)
print("FINAL PROCESSED DATASET → train_processed.csv")
print(f"Shape   : {df.shape}")
print(f"Columns : {df.columns.tolist()}")
print("=" * 60)
